# 6. Multi-Tool LangGraph Agent

Route between direct answers, weather, road distance, and flight distance.

In [ ]:
from math import asin, cos, radians, sin , sqrt 

import json

from learning_langgraph.weather import (
    GEOCODING_URL,
    fetch_json,
)

In [ ]:
def test_road_distance_normalizes_osrm_result():
    locations = {
        "Origin": {
            "name": "Origin",
            "country": "Test Country",
            "latitude": 20.0,
            "longitude": 10.0,
        },
        "Destination": {
            "name": "Destination",
            "country": "Test Country",
            "latitude": 21.0,
            "longitude": 11.0,
        },
    }

    def fake_geocoder(location):
        return locations[location]

    def fake_route_fetcher(url, params, timeout_seconds):
        return {
            "code": "Ok",
            "waypoints": [
                {
                    "name": "Origin Road",
                    "location": [10.0, 20.0],
                    "distance": 0.0,
                },
                {
                    "name": "Destination Road",
                    "location": [11.0, 21.0],
                    "distance": 0.0,
                },
            ],
            "routes": [
                {
                    "distance": 125400.0,
                    "duration": 7200.0,
                    "weight": 7200.0,
                    "weight_name": "routability",
                    "legs": [],
                }
            ],
        }

    result = get_road_distance(
        origin="Origin",
        destination="Destination",
        geocoder=fake_geocoder,
        route_fetcher=fake_route_fetcher,
    )

    assert result["status"] == "ok"
    assert result["distance_km"] == 125.4
    assert result["duration_minutes"] == 120
    assert result["source"] == "OpenStreetMap via OSRM"


def test_flight_distance_uses_great_circle_distance():
    locations = {
        "New Delhi, India": {
            "name": "New Delhi",
            "country": "India",
            "latitude": 28.6139,
            "longitude": 77.2090,
        },
        "Mumbai, India": {
            "name": "Mumbai",
            "country": "India",
            "latitude": 19.0760,
            "longitude": 72.8777,
        },
    }

    def fake_geocoder(location):
        return locations[location]

    result = get_flight_distance(
        origin="New Delhi, India",
        destination="Mumbai, India",
        geocoder=fake_geocoder,
    )

    assert result["status"] == "ok"
    assert 1140 <= result["distance_km"] <= 1160
    assert result["measurement"] == "great_circle_distance"


test_flight_distance_uses_great_circle_distance()

In [ ]:
def geocode_location(location: str) -> dict:
    payload = fetch_json(
        GEOCODING_URL,
        {
            "name": location,
            "count": 1,
            "language":"en",
            "format": "json"
        },
        10,
    )
    results = payload.get("results", [])

    if not results:
        raise ValueError(
            f"No Location found for {location}"
        )

    match = results[0]
    return {
        "name": match["name"],
        "country": match["country"],
        "latitude": match["latitude"],
        "longitude": match["longitude"],
    }

In [ ]:
def get_road_distance(
    origin: str,
    destination: str,
    geocoder=None,
    route_fetcher=None,
) -> dict:
    geocoder = geocoder or geocode_location
    route_fetcher = route_fetcher or fetch_json

    try:
        origin_location = geocoder(origin)
        destination_location = geocoder(destination)

        origin_coordinates = (
            f"{origin_location['longitude']},"
            f"{origin_location['latitude']}"
        )
        destination_coordinates = (
            f"{destination_location['longitude']},"
            f"{destination_location['latitude']}"
        )

        route_url = (
            "https://router.project-osrm.org/"
            "route/v1/driving/"
            f"{origin_coordinates};"
            f"{destination_coordinates}"
        )

        payload = route_fetcher(
            route_url,
            {
                "overview": "false",
                "steps": "false",
            },
            15,
        )

        if payload.get("code") != "Ok":
            raise ValueError(
                payload.get("message", "No road route found")
            )

        routes = payload.get("routes", [])

        if not routes:
            raise ValueError("No road route found")

        route = routes[0]

        return {
            "status": "ok",
            "origin": origin_location,
            "destination": destination_location,
            "travel_mode": "driving",
            "distance_km": round(
                route["distance"] / 1000,
                1,
            ),
            "duration_minutes": round(
                route["duration"] / 60
            ),
            "source": "OpenStreetMap via OSRM",
        }

    except Exception as error:
        return {
            "status": "error",
            "origin": origin,
            "destination": destination,
            "source": "OpenStreetMap via OSRM",
            "error": str(error),
        }

In [ ]:
def get_flight_distance(
    origin: str,
    destination: str,
    geocoder=None,
) -> dict:
    geocoder = geocoder or geocode_location

    try:
        origin_location = geocoder(origin)
        destination_location = geocoder(destination)

        origin_latitude = radians(
            origin_location["latitude"]
        )
        origin_longitude = radians(
            origin_location["longitude"]
        )
        destination_latitude = radians(
            destination_location["latitude"]
        )
        destination_longitude = radians(
            destination_location["longitude"]
        )

        latitude_difference = (
            destination_latitude - origin_latitude
        )
        longitude_difference = (
            destination_longitude - origin_longitude
        )

        haversine_value = (
            sin(latitude_difference / 2) ** 2
            + cos(origin_latitude)
            * cos(destination_latitude)
            * sin(longitude_difference / 2) ** 2
        )

        central_angle = 2 * asin(
            sqrt(haversine_value)
        )

        earth_radius_km = 6371.0088
        distance_km = earth_radius_km * central_angle

        return {
            "status": "ok",
            "origin": origin_location,
            "destination": destination_location,
            "distance_km": round(distance_km, 1),
            "measurement": "great_circle_distance",
            "source": (
                "Open-Meteo geocoding and "
                "Haversine calculation"
            ),
        }

    except Exception as error:
        return {
            "status": "error",
            "origin": origin,
            "destination": destination,
            "source": (
                "Open-Meteo geocoding and "
                "Haversine calculation"
            ),
            "error": str(error),
        }

In [ ]:
test_road_distance_normalizes_osrm_result()
test_flight_distance_uses_great_circle_distance()

print("Distance tool tests passed")

In [ ]:
road_result = get_road_distance(
    origin="Denpasar, Bali, Indonesia",
    destination="Kuta, Bali, Indonesia",
)

print(json.dumps(road_result, indent=2))

In [ ]:
flight_result = get_flight_distance(
    origin="Bengaluru, India",
    destination="Denpasar, Bali, Indonesia",
)

print(json.dumps(flight_result, indent=2))

In [ ]:
#Agent Build
import operator
import os
import time 
import json 

from datetime import date, timedelta 
from typing import Annotated, Literal, TypedDict
from langgraph.graph import END, START, StateGraph
from openai import OpenAI 
from pydantic import BaseModel

from learning_langgraph.weather import get_weather

In [ ]:
class WeatherInput(BaseModel):
    location: str
    start_date : date
    end_date: date


class RoadDistanceInput(BaseModel):
    origin: str
    destination: str

class FlightDistanceInput(BaseModel):
    origin: str
    destination: str

In [ ]:
weather_tool = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": (
            "Get live weather for a location and exact dates. "
            "Use this for weather-dependent decisions."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "Destination city and country",
                },
                "start_date": {
                    "type": "string",
                    "format": "date",
                    "description": "Start date in YYYY-MM-DD format",
                },
                "end_date": {
                    "type": "string",
                    "format": "date",
                    "description": "End date in YYYY-MM-DD format",
                },
            },
            "required": [
                "location",
                "start_date",
                "end_date",
            ],
            "additionalProperties": False,
        },
    },
}


road_distance_tool = {
    "type": "function",
    "function": {
        "name": "get_road_distance",
        "description": (
            "Get driving distance and estimated driving time "
            "between two road-connected locations. "
            "Do not use this for international air travel."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "origin": {
                    "type": "string",
                    "description": "Starting city or place",
                },
                "destination": {
                    "type": "string",
                    "description": "Destination city or place",
                },
            },
            "required": ["origin", "destination"],
            "additionalProperties": False,
        },
    },
}

flight_distance_tool = {
    "type": "function",
    "function": {
        "name": "get_flight_distance",
        "description": (
            "Estimate straight-line flight distance between "
            "cities or countries. Use this for air travel. "
            "This does not return flight schedules or prices."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "origin": {
                    "type": "string",
                    "description": "Starting city and country",
                },
                "destination": {
                    "type": "string",
                    "description": "Destination city and country",
                },
            },
            "required": ["origin", "destination"],
            "additionalProperties": False,
        },
    },
}

In [ ]:
tool_schemas = [
    weather_tool,
    road_distance_tool,
    flight_distance_tool,
]

tool_registry = {
    "get_weather": {
        "input_model": WeatherInput,
        "handler": get_weather,
    },
    "get_road_distance": {
        "input_model": RoadDistanceInput,
        "handler": get_road_distance,
    },
    "get_flight_distance": {
        "input_model": FlightDistanceInput,
        "handler": get_flight_distance,
    },
}

In [ ]:
def execute_tool_call(
    tool_call: dict,
    registry: dict,
) -> dict:
    tool_name = tool_call["function"]["name"]
    tool_id = tool_call["id"]

    tool_definition = registry.get(tool_name)

    if not tool_definition:
        return {
            "tool_call_id": tool_id,
            "tool_name": tool_name,
            "result": {
                "status": "error",
                "error": f"Tool is not allowed: {tool_name}",
            },
        }

    try:
        input_model = tool_definition["input_model"]
        validated_input = input_model.model_validate_json(
            tool_call["function"]["arguments"]
        )

        handler = tool_definition["handler"]
        result = handler(
            **validated_input.model_dump()
        )

        return {
            "tool_call_id": tool_id,
            "tool_name": tool_name,
            "result": result,
        }

    except Exception as error:
        return {
            "tool_call_id": tool_id,
            "tool_name": tool_name,
            "result": {
                "status": "error",
                "error": str(error),
            },
        }

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list[dict], operator.add]
    usage: dict
    tool_rounds: int

In [ ]:
api_key = os.getenv("AWS_BEARER_TOKEN_BEDROCK")

if not api_key:
    raise RuntimeError(
        "AWS_BEARER_TOKEN_BEDROCK is required"
    )

region = os.getenv(
    "BEDROCK_REGION",
    "ap-south-1",
)

model_id = "google.gemma-3-4b-it"

client = OpenAI(
    api_key=api_key,
    base_url=(
        f"https://bedrock-mantle.{region}.api.aws/v1"
    ),
)

In [ ]:
def empty_usage() -> dict:
    return {
        "calls": 0,
        "input_tokens": 0,
        "output_tokens": 0,
        "total_tokens": 0,
        "latency_ms": 0,
    }

def update_usage(
    current: dict,
    response,
    latency_ms: int,
) -> dict:
    call_usage = response.usage

    return {
        "calls": current["calls"] + 1,
        "input_tokens": (
            current["input_tokens"]
            + call_usage.prompt_tokens
        ),
        "output_tokens": (
            current["output_tokens"]
            + call_usage.completion_tokens
        ),
        "total_tokens": (
            current["total_tokens"]
            + call_usage.total_tokens
        ),
        "latency_ms": (
            current["latency_ms"]
            + latency_ms
        ),
    }

In [ ]:
def assistant_message_to_dict(message) -> dict:
    result = {
        "role": "assistant",
        "content": message.content,
    }

    if message.tool_calls:
        result["tool_calls"] = [
            {
                "id": tool_call.id,
                "type": "function",
                "function": {
                    "name": tool_call.function.name,
                    "arguments": (
                        tool_call.function.arguments
                    ),
                },
            }
            for tool_call in message.tool_calls
        ]

    return result

In [ ]:
def agent_node(state: AgentState) -> dict:
    request = {
        "model": model_id,
        "messages": state["messages"],
        "temperature": 0.2,
        "max_tokens": 500,
    }

    if state["tool_rounds"] < 3:
        request["tools"] = tool_schemas
        request["tool_choice"] = "auto"

    start = time.perf_counter()

    response = client.chat.completions.create(
        **request
    )

    latency_ms = round(
        (time.perf_counter() - start) * 1000
    )

    assistant_message = assistant_message_to_dict(
        response.choices[0].message
    )

    return {
        "messages": [assistant_message],
        "usage": update_usage(
            state["usage"],
            response,
            latency_ms,
        ),
    }

In [ ]:
def tool_node(state: AgentState) -> dict:
    assistant_message = state["messages"][-1]
    tool_calls = assistant_message.get(
        "tool_calls",
        [],
    )

    tool_results = [
        execute_tool_call(
            tool_call,
            tool_registry,
        )
        for tool_call in tool_calls
    ]

    result_message = {
        "role": "user",
        "content": (
            "The application executed the requested tools. "
            "Here are the exact JSON results:\n"
            f"{json.dumps(tool_results)}\n"
            "Use these results when answering. "
            "Do not invent values that are not present."
        ),
    }

    return {
        "messages": [result_message],
        "tool_rounds": state["tool_rounds"] + 1,
    }

In [ ]:
def route_after_agent(
    state: AgentState,
) -> Literal["tools", "__end__"]:
    last_message = state["messages"][-1]

    if last_message.get("tool_calls"):
        return "tools"

    return END

In [ ]:
def test_routes_tool_request_to_tools():
    state = {
        "messages": [
            {
                "role": "assistant",
                "content": None,
                "tool_calls": [
                    {
                        "id": "call-1",
                        "type": "function",
                        "function": {
                            "name": "get_road_distance",
                            "arguments": "{}",
                        },
                    }
                ],
            }
        ],
        "usage": empty_usage(),
        "tool_rounds": 0,
    }

    assert route_after_agent(state) == "tools"


def test_routes_direct_answer_to_end():
    state = {
        "messages": [
            {
                "role": "assistant",
                "content": "A direct answer",
            }
        ],
        "usage": empty_usage(),
        "tool_rounds": 0,
    }

    assert route_after_agent(state) == END


test_routes_tool_request_to_tools()
test_routes_direct_answer_to_end()

print("Graph routing tests passed")

In [ ]:
builder = StateGraph(AgentState)

builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)

builder.add_edge(START, "agent")

builder.add_conditional_edges(
    "agent",
    route_after_agent,
)

builder.add_edge(
    "tools",
    "agent",
)

graph = builder.compile()

In [ ]:
system_prompt = (
    "You are a personal decision assistant. "
    "Answer general knowledge and conversational questions directly. "
    "Use tools when a question requires live weather, road distance, "
    "driving time, or flight distance. "
    "Use the road tool only for road-connected travel. "
    "Use the flight tool for air or international travel. "
    "You may request more than one tool when a decision needs "
    "multiple facts. "
    "Never claim that a tool was used unless you requested it. "
    "Never invent tool results. "
    "Mention data sources and important limitations."
)

def run_agent(question: str) -> dict:
    initial_state = {
        "messages": [
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": question,
            },
        ],
        "usage": empty_usage(),
        "tool_rounds": 0,
    }

    return graph.invoke(
        initial_state,
        config={"recursion_limit": 8},
    )

In [ ]:
result = run_agent(
    "What are three benefits of planning a trip early?"
)

print(result["messages"][-1]["content"])
print(json.dumps(result["usage"], indent=2))

In [ ]:
requested_tools = [
    tool_call["function"]["name"]
    for message in result["messages"]
    for tool_call in message.get("tool_calls", [])
]

print("Requested tools:", requested_tools)

In [ ]:
result = run_agent(
    "How far is Denpasar from Kuta by road, "
    "and approximately how long is the drive?"
)

print(result["messages"][-1]["content"])
print(json.dumps(result["usage"], indent=2))

In [ ]:
trip_start = date.today() + timedelta(days=3)
trip_end = trip_start + timedelta(days=3)

question = (
    f"I am considering a trip from Bengaluru, India "
    f"to Denpasar, Bali, Indonesia from "
    f"{trip_start} to {trip_end}. "
    "Check the flight distance and weather. "
    "Then tell me whether the trip seems reasonable."
)

result = run_agent(question)

print("Final answer:")
print(result["messages"][-1]["content"])

print("\nUsage:")
print(json.dumps(result["usage"], indent=2))

print("\nConversation:")
for index, message in enumerate(result["messages"]):
    print(f"\nMessage {index}")
    print(json.dumps(message, indent=2))

## Known model limitation

With `tool_choice="auto"`, Gemma 3 may write text that looks like a tool call without returning a real `tool_calls` object. The graph can execute only structured tool calls returned by the SDK. Forced tool selection is more reliable for testing.